In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql import types as T


# -----------------------------------------------
# 1. Read pipeline configuration
# -----------------------------------------------

EH_NAMESPACE = spark.conf.get(
    "orders.ingestion.eh.namespace"
)
EH_NAME = spark.conf.get(
    "orders.ingestion.eh.name"
)
ACCESS_KEY_NAME = spark.conf.get(
    "orders.ingestion.eh.accessKeyName"
)
SECRET_SCOPE = spark.conf.get(
    "orders.ingestion.eh.secretsScopeName"
)

ACCESS_KEY = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key=ACCESS_KEY_NAME
)

EH_CONNECTION_STRING = (
    f"Endpoint=sb://{EH_NAMESPACE}.servicebus.windows.net/;"
    f"SharedAccessKeyName={ACCESS_KEY_NAME};"
    f"SharedAccessKey={ACCESS_KEY}"
)


# -----------------------------------------------
# 2. Configure the Kafka-compatible connection
# -----------------------------------------------

KAFKA_OPTIONS = {
    "kafka.bootstrap.servers":
        f"{EH_NAMESPACE}.servicebus.windows.net:9093",
    "subscribe": EH_NAME,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        "kafkashaded.org.apache.kafka.common.security.plain."
        "PlainLoginModule required "
        'username="$ConnectionString" '
        f'password="{EH_CONNECTION_STRING}";'
    ),
    "kafka.request.timeout.ms": spark.conf.get(
        "orders.ingestion.kafka.requestTimeout"
    ),
    "kafka.session.timeout.ms": spark.conf.get(
        "orders.ingestion.kafka.sessionTimeout"
    ),
    "maxOffsetsPerTrigger": spark.conf.get(
        "orders.ingestion.spark.maxOffsetsPerTrigger"
    ),
    "startingOffsets": spark.conf.get(
        "orders.ingestion.spark.startingOffsets"
    ),
    "failOnDataLoss": spark.conf.get(
        "orders.ingestion.spark.failOnDataLoss"
    )
}


# -----------------------------------------------
# 3. Bronze: store raw events
# -----------------------------------------------

@dp.table(
    name="orders_bronze",
    comment="Raw order events ingested from Azure Event Hubs"
)
def orders_bronze():

    return (
        spark.readStream
        .format("kafka")
        .options(**KAFKA_OPTIONS)
        .load()
        .select(
            F.col("value")
                .cast("string")
                .alias("event_json"),
            F.col("timestamp")
                .alias("eh_enqueued_timestamp"),
            F.col("topic").alias("eh_topic"),
            F.col("partition").alias("eh_partition"),
            F.col("offset").alias("eh_offset"),
            F.current_timestamp()
                .alias("ingest_timestamp")
        )
    )


# -----------------------------------------------
# 4. Define the JSON event schema
# -----------------------------------------------

ORDER_SCHEMA = T.StructType([
    T.StructField("event_id", T.StringType(), True),
    T.StructField("order_id", T.StringType(), True),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("product_id", T.StringType(), True),
    T.StructField("quantity", T.IntegerType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("event_time", T.StringType(), True)
])


# -----------------------------------------------
# 5. Silver: parse and validate events
# -----------------------------------------------

@dp.table(
    name="orders_silver",
    comment="Parsed and validated order events"
)
@dp.expect_or_drop(
    "valid_order",
    """
    event_id IS NOT NULL
    AND order_id IS NOT NULL
    AND quantity > 0
    AND amount >= 0
    AND event_time IS NOT NULL
    """
)
def orders_silver():

    raw_stream = spark.readStream.table(
        "orders_bronze"
    )

    parsed = raw_stream.withColumn(
        "order_data",
        F.from_json(
            F.col("event_json"),
            ORDER_SCHEMA
        )
    )

    orders = parsed.select(
        F.col("order_data.event_id").alias("event_id"),
        F.col("order_data.order_id").alias("order_id"),
        F.col("order_data.customer_id").alias("customer_id"),
        F.col("order_data.product_id").alias("product_id"),
        F.col("order_data.quantity").alias("quantity"),
        F.col("order_data.amount").alias("amount"),
        F.col("order_data.status").alias("status"),
        F.to_timestamp(
            F.col("order_data.event_time")
        ).alias("event_time"),
        F.col("eh_enqueued_timestamp"),
        F.col("ingest_timestamp"),
        F.col("eh_topic"),
        F.col("eh_partition"),
        F.col("eh_offset")
    )

    return (
        orders
        .filter(F.col("event_time").isNotNull())
        .withWatermark("event_time", "10 minutes")
        .dropDuplicates(["event_id", "event_time"])
    )